In [1]:
pip install youtube-transcript-api


   -------------------- ------------------- 1/2 [youtube-transcript-api]
   ---------------------------------------- 2/2 [youtube-transcript-api]

Note: you may need to restart the kernel to use updated packages.


In [49]:
pip install yt-dlp

Note: you may need to restart the kernel to use updated packages.


In [31]:
import os
from dotenv import load_dotenv
load_dotenv()
print(os.getenv("GOOGLE_API_KEY")[:6] + "...")

AQ.Ab8...


In [64]:
import inspect
from youtube_transcript_api import YouTubeTranscriptApi

print(inspect.signature(YouTubeTranscriptApi.__init__))

(self, proxy_config: Optional[youtube_transcript_api.proxies.ProxyConfig] = None, http_client: Optional[requests.sessions.Session] = None)


In [81]:
import requests
import http.cookiejar
from youtube_transcript_api import YouTubeTranscriptApi

url = "https://www.youtube.com/watch?v=r-7igwlC1kk"
video_id = url.split("youtu.be/")[-1].split("?")[0]

try:
    # Load cookies into a requests session
    session = requests.Session()
    cookies = http.cookiejar.MozillaCookieJar("cookies.txt")
    cookies.load()
    session.cookies = cookies

    # Pass session to API
    ytt = YouTubeTranscriptApi(http_client=session)
    transcript_list = ytt.fetch(video_id)
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript[:500])

except Exception as e:
    print(f"Error: {e}")

Error: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=https://www.youtube.com/watch! This is most likely caused by:

You provided an invalid video id. Make sure you are using the video id and NOT the url!

Do NOT run: `YouTubeTranscriptApi().fetch("https://www.youtube.com/watch?v=1234")`
Instead run: `YouTubeTranscriptApi().fetch("1234")`

If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide the information needed to replicate the error. Also make sure that there are no open issues which already describe your problem!


In [82]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='relax', start=1.48, duration=6.2), FetchedTranscriptSnippet(text="breathe live you know like we're alive", start=3.0, duration=7.599), FetchedTranscriptSnippet(text="that's a pretty cool", start=7.68, duration=5.72), FetchedTranscriptSnippet(text='thing', start=10.599, duration=5.361), FetchedTranscriptSnippet(text="um don't worry so", start=13.4, duration=5.84), FetchedTranscriptSnippet(text='much things always work out as they', start=15.96, duration=6.04), FetchedTranscriptSnippet(text='should and I know like as humans we want', start=19.24, duration=5.16), FetchedTranscriptSnippet(text='to hyperfixate and over analy and', start=22.0, duration=5.64), FetchedTranscriptSnippet(text='micromanage everything in our lives but', start=24.4, duration=4.84), FetchedTranscriptSnippet(text="most of it's just out of our control", start=27.64, duration=4.759), FetchedTranscriptSnippet(text="anyways and US worrying about it doesn't", star

In [83]:
transcript

"relax breathe live you know like we're alive that's a pretty cool thing um don't worry so much things always work out as they should and I know like as humans we want to hyperfixate and over analy and micromanage everything in our lives but most of it's just out of our control anyways and US worrying about it doesn't do anything doesn't do anything it just makes our lives worse takes away from our ability to laugh and enjoy what we have be grateful you have so much to be grateful for life is beautiful man the process is beautiful even when you were lost when you don't have it all figured out that's when you grow gr and develop and that is where Adventure lies if things were always certain if things if there was no chance no risk no unknown how boring would that be that's not an existence I want to live you have to have some of that in there at least a little bit spice things up we need an adventure you know because cuz man those moments when you get through the thick of it and you sta

step 02 - chunking

In [84]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Wrap as LangChain Document
doc = Document(
    page_content=transcript,
    metadata={
        "source": url,
        "video_id": video_id
    }
)

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks = splitter.split_documents([doc])

print(f"Total chunks: {len(chunks)}")
print("\n--- First chunk preview ---")
print(chunks[0].page_content)

Total chunks: 4

--- First chunk preview ---
relax breathe live you know like we're alive that's a pretty cool thing um don't worry so much things always work out as they should and I know like as humans we want to hyperfixate and over analy and micromanage everything in our lives but most of it's just out of our control anyways and US worrying about it doesn't do anything doesn't do anything it just makes our lives worse takes away from our ability to laugh and enjoy what we have be grateful you have so much to be grateful for life is


In [85]:
len(chunks)

4

In [86]:
chunks[-1]

Document(metadata={'source': 'https://www.youtube.com/watch?v=r-7igwlC1kk', 'video_id': 'https://www.youtube.com/watch'}, page_content="you and you say I can't wait that's what this is about it's not about paralyzing yourself with worry and fear about things that you can't control anyways so as you march toward that mountain and enter this chapter in your life or just go about your day do it with a smile do it with a smile")

step 03 - emedding & vectoization

In [87]:
import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [88]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

print("Vector store ready!")
print(f"Total vectors indexed: {vector_store.index.ntotal}")

Vector store ready!
Total vectors indexed: 4


step 04 - RAG

In [89]:
import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [90]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# LLM
llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0.3)

# Prompt
prompt = PromptTemplate(
    template="""
    You are a helpful assistant that answers questions about a YouTube video.
    Answer ONLY from the provided transcript context.
    If the context is insufficient, just say you don't know.

    Context: {context}
    Question: {question}
    """,
    input_variables=["context", "question"]
)

# Format retrieved docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
}) | prompt | llm | StrOutputParser()

print("Chain ready!")

Chain ready!


In [91]:
response = chain.invoke("What advice does the video give about worry?")
print(response)

The video advises not to worry so much because things always work out as they should. It states that worrying doesn't do anything positive and just makes our lives worse, taking away from our ability to laugh and enjoy what we have. It also suggests not paralyzing oneself with worry and fear about things that can't be controlled.


In [92]:
questions = [
    "What advice does the video give about worry?",
    "How should we deal with challenges according to the video?",
    "What does the video say about gratitude?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {chain.invoke(q)}")
    print("-" * 50)

Q: What advice does the video give about worry?
A: The video advises not to worry so much because things always work out as they should. It also states that worrying doesn't do anything, makes our lives worse, and takes away from our ability to laugh and enjoy what we have. It emphasizes not paralyzing oneself with worry and fear about things that can't be controlled.
--------------------------------------------------
Q: How should we deal with challenges according to the video?
A: According to the video, when facing challenges or "mountains," you should not paralyze yourself with worry and fear about things you can't control. Instead, you should march toward them with a smile. The speaker also suggests embracing the process, even when lost, as that's when you grow and develop, and where adventure lies. It's important not to worry so much, as things often work out as they should, and worrying only makes life worse and takes away from your ability to laugh and enjoy what you have. You s